In [2]:
pip install pynput pandas openpyxl


Defaulting to user installation because normal site-packages is not writeable

   ---------------------------------------- 0/3 [pynput]
   ---------------------------------------- 0/3 [pynput]
   ---------------------------------------- 0/3 [pynput]
   ------------- -------------------------- 1/3 [et-xmlfile]
   -------------------------- ------------- 2/3 [openpyxl]
   -------------------------- ------------- 2/3 [openpyxl]
   -------------------------- ------------- 2/3 [openpyxl]
   -------------------------- ------------- 2/3 [openpyxl]
   -------------------------- ------------- 2/3 [openpyxl]
   -------------------------- ------------- 2/3 [openpyxl]
   -------------------------- ------------- 2/3 [openpyxl]
   -------------------------- ------------- 2/3 [openpyxl]
   -------------------------- ------------- 2/3 [openpyxl]
   -------------------------- ------------- 2/3 [openpyxl]
   -------------------------- ------------- 2/3 [openpyxl]
   -------------------------- ----------


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: C:\Users\gorip\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [ ]:
# app.py
"""
Tkinter UI that:
 - Login page (enter participant id and password)
 -enter into website after login
 then show
 - Questions page (6 questions)
 - Submit page (review + save)
 -  After click on submit ,inside portal  it should track mouse/keyboard/idel activity  in the background
 - save all clicked events and non clicked events that is called idel time  with timestamps to an Excel file only their actions (I don't need their id and their answers to the questions).

Dependencies:
    pip install pynput pandas openpyxl
Run:
    python app.py
"""

import threading
import time
import os
import sys
from datetime import datetime
import tkinter as tk
from tkinter import ttk, messagebox
from pynput import mouse, keyboard
import pandas as pd

# -------- Config ----------
OUTPUT_FILE = "activity_log.xlsx"
IDLE_THRESHOLD = 5.0  # seconds: gap greater than this counts as idle period
# --------------------------

# Shared structures to store events and answers
event_log = []   # list of dicts: {timestamp, event_type, detail, x, y}
answers = {}     # question_index -> answer
session_meta = {}  # participant_id, start_time, end_time

# For idle calculation
_last_event_time = None
_total_idle_time = 0.0
_idle_periods = []  # list of (idle_start, idle_end, duration)


# ---------- pynput listeners ----------
def on_click(x, y, button, pressed):
    """Mouse callback."""
    global _last_event_time, _total_idle_time
    ts = datetime.utcnow()
    ev = {
        "timestamp_utc": ts,
        "event_type": "mouse_click" if pressed else "mouse_release",
        "detail": str(button),
        "x": x,
        "y": y
    }
    _handle_event_time(ts)
    event_log.append(ev)


def on_scroll(x, y, dx, dy):
    global _last_event_time, _total_idle_time
    ts = datetime.utcnow()
    ev = {
        "timestamp_utc": ts,
        "event_type": "mouse_scroll",
        "detail": f"dx={dx},dy={dy}",
        "x": x,
        "y": y
    }
    _handle_event_time(ts)
    event_log.append(ev)


def on_press(key):
    global _last_event_time, _total_idle_time
    ts = datetime.utcnow()
    try:
        detail = key.char
    except AttributeError:
        detail = str(key)
    ev = {
        "timestamp_utc": ts,
        "event_type": "key_press",
        "detail": detail,
        "x": None,
        "y": None
    }
    _handle_event_time(ts)
    event_log.append(ev)


def _handle_event_time(ts):
    """Update idle tracking based on arrival time ts (datetime)."""
    global _last_event_time, _total_idle_time, _idle_periods
    if _last_event_time is None:
        _last_event_time = ts
        return
    gap = (ts - _last_event_time).total_seconds()
    if gap > IDLE_THRESHOLD:
        # record idle period
        idle_start = _last_event_time
        idle_end = ts
        dur = gap
        _idle_periods.append((idle_start, idle_end, dur))
        _total_idle_time += dur
    _last_event_time = ts


class InputListeners:
    def __init__(self):
        self.mouse_listener = None
        self.keyboard_listener = None
        self.running = False

    def start(self):
        if self.running:
            return
        # start listeners in separate threads
        self.mouse_listener = mouse.Listener(on_click=on_click, on_scroll=on_scroll)
        self.keyboard_listener = keyboard.Listener(on_press=on_press)
        self.mouse_listener.start()
        self.keyboard_listener.start()
        self.running = True

    def stop(self):
        if not self.running:
            return
        try:
            if self.mouse_listener:
                self.mouse_listener.stop()
                self.mouse_listener.join(timeout=1)
            if self.keyboard_listener:
                self.keyboard_listener.stop()
                self.keyboard_listener.join(timeout=1)
        except Exception:
            pass
        self.running = False


listeners = InputListeners()

# ---------- Tkinter UI ----------
class App(tk.Tk):
    def __init__(self):
        super().__init__()
        self.title("Activity Capture App")
        self.geometry("700x520")
        self.protocol("WM_DELETE_WINDOW", self.on_close)

        # Stacked frames
        self.container = ttk.Frame(self)
        self.container.pack(fill="both", expand=True)
        self.frames = {}

        for F in (LoginPage, QuestionsPage, SubmitPage):
            page_name = F.__name__
            frame = F(parent=self.container, controller=self)
            self.frames[page_name] = frame
            frame.grid(row=0, column=0, sticky="nsew")

        self.show_frame("LoginPage")

    def show_frame(self, page_name):
        frame = self.frames[page_name]
        frame.tkraise()

    def on_close(self):
        # Stop listeners
        if messagebox.askokcancel("Quit", "Do you want to quit?"):
            try:
                listeners.stop()
            except Exception:
                pass
            # finalize session end time
            session_meta.setdefault("end_time_utc", datetime.utcnow())
            # auto-save before exit?
            # Ask user to explicitly save in Submit page. But still try a quick save of event log for debug.
            try:
                save_quick_log()
            except Exception:
                pass
            self.destroy()


class LoginPage(ttk.Frame):
    def __init__(self, parent, controller):
        super().__init__(parent, padding=20)
        self.controller = controller

        ttk.Label(self, text="Login", font=("Helvetica", 18, "bold")).pack(pady=10)
        ttk.Label(self, text="Participant ID:").pack(pady=(10, 0))
        self.participant_var = tk.StringVar()
        ttk.Entry(self, textvariable=self.participant_var, width=40).pack(pady=5)

        ttk.Label(self, text="Session Notes (optional):").pack(pady=(10, 0))
        self.notes_text = tk.Text(self, height=4)
        self.notes_text.pack(pady=5, fill="x")

        btn_frame = ttk.Frame(self)
        btn_frame.pack(pady=20)
        ttk.Button(btn_frame, text="Start Session", command=self.start_session).pack()

    def start_session(self):
        pid = self.participant_var.get().strip()
        notes = self.notes_text.get("1.0", "end").strip()
        if not pid:
            messagebox.showerror("Missing Participant ID", "Please enter a Participant ID to continue.")
            return
        session_meta["participant_id"] = pid
        session_meta["notes"] = notes
        session_meta["start_time_utc"] = datetime.utcnow()
        # start listeners
        try:
            listeners.start()
        except Exception as e:
            messagebox.showwarning("Listener error", f"Failed to start input listeners: {e}")
        self.controller.show_frame("QuestionsPage")


QUESTIONS = [
    "1. What is your age group?",
    "2. Have you used similar apps before? (Yes/No)",
    "3. Rate your familiarity with computers (1-5).",
    "4. How comfortable are you with typing? (1-5).",
    "5. What is your primary purpose for participating?",
    "6. Any additional comments or feedback?"
]


class QuestionsPage(ttk.Frame):
    def __init__(self, parent, controller):
        super().__init__(parent, padding=12)
        self.controller = controller
        ttk.Label(self, text="Questions", font=("Helvetica", 16, "bold")).pack(pady=8)

        self.entries = {}
        for i, q in enumerate(QUESTIONS, start=1):
            frame = ttk.Frame(self)
            frame.pack(fill="x", pady=6)
            ttk.Label(frame, text=q).pack(anchor="w")
            if i == 5 or i == 6:
                txt = tk.Text(frame, height=3)
                txt.pack(fill="x")
                self.entries[i] = txt
            else:
                sv = tk.StringVar()
                ent = ttk.Entry(frame, textvariable=sv)
                ent.pack(fill="x")
                self.entries[i] = sv

        nav = ttk.Frame(self)
        nav.pack(pady=15)
        ttk.Button(nav, text="Previous (Login)", command=lambda: controller.show_frame("LoginPage")).pack(side="left", padx=8)
        ttk.Button(nav, text="Next (Submit)", command=self.go_to_submit).pack(side="left", padx=8)

    def go_to_submit(self):
        # collect answers
        for i in range(1, len(QUESTIONS) + 1):
            widget = self.entries[i]
            if isinstance(widget, tk.StringVar):
                val = widget.get().strip()
            else:
                val = widget.get("1.0", "end").strip()
            answers[f"Q{i}"] = val
        session_meta["questions_start_utc"] = session_meta.get("questions_start_utc", datetime.utcnow())
        self.controller.show_frame("SubmitPage")


class SubmitPage(ttk.Frame):
    def __init__(self, parent, controller):
        super().__init__(parent, padding=12)
        self.controller = controller
        ttk.Label(self, text="Submit & Save", font=("Helvetica", 16, "bold")).pack(pady=8)

        self.summary = tk.Text(self, height=15)
        self.summary.pack(fill="both", expand=True, pady=6)
        btn_frame = ttk.Frame(self)
        btn_frame.pack(pady=8)
        ttk.Button(btn_frame, text="Refresh Summary", command=self.refresh).pack(side="left", padx=6)
        ttk.Button(btn_frame, text="Save to Excel", command=self.save).pack(side="left", padx=6)
        ttk.Button(btn_frame, text="Finish & Exit", command=self.finish).pack(side="left", padx=6)
        self.refresh()

    def refresh(self):
        self.summary.delete("1.0", "end")
        lines = []
        lines.append(f"Participant ID: {session_meta.get('participant_id','-')}")
        lines.append(f"Session start (UTC): {session_meta.get('start_time_utc')}")
        lines.append(f"Session end (UTC): {session_meta.get('end_time_utc','-')}")
        lines.append("")
        lines.append("Answers:")
        for i in range(1, len(QUESTIONS) + 1):
            lines.append(f"Q{i}: {answers.get(f'Q{i}','')}")
        lines.append("")
        lines.append("Event summary:")
        lines.append(f"Total events captured: {len(event_log)}")
        lines.append(f"Total idle time (seconds): {round(_total_idle_time,3)}")
        lines.append(f"Idle periods count: {len(_idle_periods)}")
        lines.append("")
        lines.append("Last 10 events (most recent last):")
        for ev in event_log[-10:]:
            lines.append(f"{ev['timestamp_utc']} | {ev['event_type']} | {ev['detail']} | pos=({ev.get('x')},{ev.get('y')})")
        self.summary.insert("1.0", "\n".join(lines))

    def save(self):
        # stop listeners to finalize timestamps
        listeners.stop()
        session_meta.setdefault("end_time_utc", datetime.utcnow())

        try:
            save_session_to_excel()
            messagebox.showinfo("Saved", f"Session saved to {OUTPUT_FILE}")
        except Exception as e:
            messagebox.showerror("Save error", f"Failed to save: {e}")

    def finish(self):
        # Ensure saved
        try:
            save_session_to_excel()
        except Exception:
            pass
        self.controller.on_close()


# ---------- Save logic ----------
def _build_session_dataframe():
    """Build a DataFrame capturing per-event rows joined with session info and answers."""
    rows = []
    for ev in event_log:
        row = {
            "participant_id": session_meta.get("participant_id"),
            "notes": session_meta.get("notes"),
            "session_start_utc": session_meta.get("start_time_utc"),
            "session_end_utc": session_meta.get("end_time_utc"),
            "event_timestamp_utc": ev["timestamp_utc"],
            "event_type": ev["event_type"],
            "detail": ev.get("detail"),
            "x": ev.get("x"),
            "y": ev.get("y"),
            "total_idle_time_seconds": _total_idle_time,
            "idle_periods_count": len(_idle_periods)
        }
        # add answers as separate columns
        for i in range(1, len(QUESTIONS) + 1):
            row[f"Q{i}"] = answers.get(f"Q{i}", "")
        rows.append(row)
    # If there were no events, still create a single summary row
    if not rows:
        row = {
            "participant_id": session_meta.get("participant_id"),
            "notes": session_meta.get("notes"),
            "session_start_utc": session_meta.get("start_time_utc"),
            "session_end_utc": session_meta.get("end_time_utc"),
            "event_timestamp_utc": None,
            "event_type": None,
            "detail": None,
            "x": None,
            "y": None,
            "total_idle_time_seconds": _total_idle_time,
            "idle_periods_count": len(_idle_periods)
        }
        for i in range(1, len(QUESTIONS) + 1):
            row[f"Q{i}"] = answers.get(f"Q{i}", "")
        rows.append(row)
    df = pd.DataFrame(rows)
    # Normalize timestamp columns to ISO strings for safe Excel writing
    df["session_start_utc"] = df["session_start_utc"].apply(lambda x: x.isoformat() if pd.notna(x) else "")
    df["session_end_utc"] = df["session_end_utc"].apply(lambda x: x.isoformat() if pd.notna(x) else "")
    df["event_timestamp_utc"] = df["event_timestamp_utc"].apply(lambda x: x.isoformat() if pd.notna(x) else "")
    return df


def save_session_to_excel():
    """Append current session dataframe to OUTPUT_FILE. If exists, append rows."""
    df = _build_session_dataframe()
    if os.path.exists(OUTPUT_FILE):
        try:
            existing = pd.read_excel(OUTPUT_FILE)
            combined = pd.concat([existing, df], ignore_index=True, sort=False)
            combined.to_excel(OUTPUT_FILE, index=False)
        except Exception as e:
            # fallback: try writing to a new file with timestamp
            alt = f"activity_log_{int(time.time())}.xlsx"
            df.to_excel(alt, index=False)
            raise RuntimeError(f"Could not append to {OUTPUT_FILE}: {e}. Saved to {alt} instead.")
    else:
        df.to_excel(OUTPUT_FILE, index=False)


def save_quick_log():
    """Quick dump of events only (for debugging on forced exit)."""
    if not event_log:
        return
    rows = []
    for ev in event_log:
        rows.append({
            "timestamp_utc": ev["timestamp_utc"].isoformat(),
            "event_type": ev["event_type"],
            "detail": ev.get("detail"),
            "x": ev.get("x"),
            "y": ev.get("y")
        })
    df = pd.DataFrame(rows)
    tmp = f"quick_event_dump_{int(time.time())}.xlsx"
    df.to_excel(tmp, index=False)


# ---------- Entrypoint ----------
def main():
    # Basic check: pynput available
    try:
        import pynput  # noqa: F401
    except Exception as e:
        print("pynput is required. Install with: pip install pynput")
        sys.exit(1)

    app = App()
    app.mainloop()


if __name__ == "__main__":
    main()


C:\Users\gorip\AppData\Local\Temp\ipykernel_17660\4239866279.py:211: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  session_meta["start_time_utc"] = datetime.utcnow()
C:\Users\gorip\AppData\Local\Temp\ipykernel_17660\4239866279.py:62: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  ts = datetime.utcnow()
C:\Users\gorip\AppData\Local\Temp\ipykernel_17660\4239866279.py:48: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  ts = datetime.utcnow()
C:\Users\gorip\AppData\Local\Temp\ipykernel_17660\4239866279.py:76: DeprecationWarn

In [2]:
import tkinter as tk
from tkinter import ttk, messagebox
import pandas as pd
import webbrowser
from datetime import datetime
import time
import os
from pynput import mouse, keyboard

OUTPUT_FILE = "activity_log.xlsx"
IDLE_THRESHOLD = 5  # seconds

event_log = []
idle_periods = []

_last_event_time = None
_total_idle_time = 0

def record_event(event_type, detail=None, x=None, y=None):
    global _last_event_time, _total_idle_time

    ts = datetime.utcnow()

    # Idle tracking
    if _last_event_time is not None:
        gap = (ts - _last_event_time).total_seconds()
        if gap > IDLE_THRESHOLD:
            idle_periods.append({
                "idle_start_utc": _last_event_time.isoformat(),
                "idle_end_utc": ts.isoformat(),
                "duration_seconds": gap
            })
            _total_idle_time += gap

    _last_event_time = ts

    event_log.append({
        "timestamp_utc": ts.isoformat(),
        "event_type": event_type,
        "detail": detail,
        "x": x,
        "y": y
    })


# Mouse listener
def on_click(x, y, button, pressed):
    if pressed:
        record_event("mouse_click", str(button), x, y)

def on_scroll(x, y, dx, dy):
    record_event("mouse_scroll", f"dx={dx},dy={dy}", x, y)


# Keyboard listener
def on_key_press(key):
    try:
        detail = key.char
    except:
        detail = str(key)
    record_event("key_press", detail)


class ActivityApp(tk.Tk):
    def __init__(self):
        super().__init__()
        self.title("Activity Logger")
        self.geometry("600x420")
        self.resizable(False, False)

        self.container = ttk.Frame(self)
        self.container.pack(fill="both", expand=True)

        self.frames = {}
        for F in (LoginPage, QuestionsPage, SubmitPage):
            frame = F(self.container, self)
            self.frames[F.__name__] = frame
            frame.grid(row=0, column=0, sticky="nsew")

        self.show("LoginPage")

    def show(self, name):
        self.frames[name].tkraise()


class LoginPage(ttk.Frame):
    def __init__(self, parent, controller):
        super().__init__(parent, padding=20)
        self.controller = controller

        ttk.Label(self, text="Login", font=("Arial", 18, "bold")).pack(pady=10)
        ttk.Button(self, text="Next", command=lambda: controller.show("QuestionsPage")).pack(pady=20)


class QuestionsPage(ttk.Frame):
    def __init__(self, parent, controller):
        super().__init__(parent, padding=20)
        self.controller = controller

        ttk.Label(self, text="Questions", font=("Arial", 16, "bold")).pack(pady=10)

        self.entries = []
        q_text = [
            "Q1:", "Q2:", "Q3:", "Q4:", "Q5:", "Q6:"
        ]

        for label in q_text:
            frame = ttk.Frame(self)
            frame.pack(fill="x", pady=5)
            ttk.Label(frame, text=label).pack(side="left")
            entry = ttk.Entry(frame, width=40)
            entry.pack(side="left")
            self.entries.append(entry)

        ttk.Button(self, text="Submit", command=self.go_submit).pack(pady=20)

    def go_submit(self):
        self.controller.show("SubmitPage")


class SubmitPage(ttk.Frame):
    def __init__(self, parent, controller):
        super().__init__(parent, padding=20)
        self.controller = controller

        ttk.Label(self, text="Submit Page", font=("Arial", 16, "bold")).pack(pady=10)

        ttk.Button(self, text="Open Website & Start Tracking", command=self.start_tracking).pack(pady=10)
        ttk.Button(self, text="Save Excel", command=self.save_to_excel).pack(pady=10)
        ttk.Button(self, text="Exit", command=self.exit_app).pack(pady=10)

    def start_tracking(self):
        # Example website — change as needed
        url = "https://www.irctc.co.in"
        webbrowser.open(url)

        # Start background listeners
        mouse_listener = mouse.Listener(on_click=on_click, on_scroll=on_scroll)
        keyboard_listener = keyboard.Listener(on_press=on_key_press)

        mouse_listener.start()
        keyboard_listener.start()

        messagebox.showinfo("Tracking Started", "Mouse, keyboard, and idle time tracking started.")

    def save_to_excel(self):
        global event_log, idle_periods, _total_idle_time

        df_events = pd.DataFrame(event_log)
        df_idle = pd.DataFrame(idle_periods)

        with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl", mode="w") as writer:
            df_events.to_excel(writer, index=False, sheet_name="Events")
            df_idle.to_excel(writer, index=False, sheet_name="IdleTime")

        messagebox.showinfo("Saved", f"Saved to {OUTPUT_FILE}")

    def exit_app(self):
        self.controller.destroy()


if __name__ == "__main__":
    app = ActivityApp()
    app.mainloop()


In [ ]:
"""
Updated Tkinter App
------------------------------------
Flow:
Login Page → Questions Page → Submit Page → On Submit:
- Opens a website (IRCTC or any URL)
- Starts tracking mouse clicks, key presses, idle time (background)
- Saves ONLY events + idle timestamps in Excel

NOT saving questions or answers in Excel
NOT saving Q/A pages data

Dependencies:
    pip install pynput pandas openpyxl

Run:
    python app.py
"""

import tkinter as tk
from tkinter import ttk, messagebox
from datetime import datetime
import time
import threading
import webbrowser
import pandas as pd
import os

from pynput import mouse, keyboard

# ========== CONFIG ===========
OUTPUT_FILE = "activity_log.xlsx"
TRACKING_URL = "https://www.irctc.co.in"         # <-- change if needed
IDLE_THRESHOLD = 5.0                             # seconds
# =============================

event_log = []
_last_event_time = None
_idle_periods = []
_tracking_started = False


# ----------- EVENT HANDLERS -----------
def _record_event(event_type, detail=None, x=None, y=None):
    global _last_event_time, _idle_periods

    ts = datetime.utcnow()

    # idle detection
    if _last_event_time:
        gap = (ts - _last_event_time).total_seconds()
        if gap > IDLE_THRESHOLD:
            # record idle event instead of storing multiple rows
            event_log.append({
                "timestamp": ts.isoformat(),
                "event_type": "idle_period",
                "detail": f"idle_for_{gap:.2f}_seconds",
                "x": None,
                "y": None
            })
    _last_event_time = ts

    # record actual event
    event_log.append({
        "timestamp": ts.isoformat(),
        "event_type": event_type,
        "detail": detail,
        "x": x,
        "y": y
    })


def on_click(x, y, button, pressed):
    if pressed:
        _record_event("mouse_click", str(button), x, y)


def on_press(key):
    try:
        detail = key.char
    except AttributeError:
        detail = str(key)
    _record_event("key_press", detail, None, None)


# --------- LISTENER THREAD START ----------
def start_tracking_background():
    global _tracking_started
    if _tracking_started:
        return

    _tracking_started = True

    # start listeners in background threads
    threading.Thread(target=lambda: mouse.Listener(on_click=on_click).run(), daemon=True).start()
    threading.Thread(target=lambda: keyboard.Listener(on_press=on_press).run(), daemon=True).start()


# --------- SAVE TO EXCEL ----------
def save_events_to_excel():
    df = pd.DataFrame(event_log)

    if os.path.exists(OUTPUT_FILE):
        old = pd.read_excel(OUTPUT_FILE)
        df = pd.concat([old, df], ignore_index=True)

    df.to_excel(OUTPUT_FILE, index=False)


# --------------------------------------------------
#                 TKINTER UI
# --------------------------------------------------

class App(tk.Tk):
    def __init__(self):
        super().__init__()
        self.title("Activity Capture App")
        self.geometry("600x450")

        self.container = ttk.Frame(self)
        self.container.pack(fill="both", expand=True)

        self.frames = {}

        for F in (LoginPage, QuestionsPage, SubmitPage):
            frame = F(parent=self.container, controller=self)
            self.frames[F.__name__] = frame
            frame.grid(row=0, column=0, sticky="nsew")

        self.show("LoginPage")

    def show(self, page):
        frame = self.frames[page]
        frame.tkraise()


# ---------------- LOGIN PAGE ----------------
class LoginPage(ttk.Frame):
    def __init__(self, parent, controller):
        super().__init__(parent, padding=20)
        self.controller = controller

        ttk.Label(self, text="Login", font=("Arial", 18, "bold")).pack(pady=10)

        ttk.Label(self, text="Participant ID:").pack()
        self.pid = tk.StringVar()
        ttk.Entry(self, textvariable=self.pid).pack(pady=5)

        ttk.Button(self, text="Next", command=self.next).pack(pady=20)

    def next(self):
        if not self.pid.get().strip():
            messagebox.showerror("Error", "Enter Participant ID")
            return

        self.controller.show("QuestionsPage")


# ---------------- QUESTIONS PAGE ----------------
class QuestionsPage(ttk.Frame):
    QUESTIONS = [
        "1. What is your age group?",
        "2. Have you used similar apps before?",
        "3. Rate your familiarity with computers (1-5).",
        "4. How comfortable are you with typing? (1-5).",
        "5. What is your purpose for participating?",
        "6. Any additional feedback?"
    ]

    def __init__(self, parent, controller):
        super().__init__(parent, padding=15)
        self.controller = controller

        ttk.Label(self, text="Questions", font=("Arial", 16, "bold")).pack()

        self.inputs = {}

        for q in self.QUESTIONS:
            frame = ttk.Frame(self)
            frame.pack(fill="x", pady=6)

            ttk.Label(frame, text=q).pack(anchor="w")
            ent = ttk.Entry(frame)
            ent.pack(fill="x")
            self.inputs[q] = ent

        ttk.Button(self, text="Submit", command=self.go_to_submit).pack(pady=20)

    def go_to_submit(self):
        # NOT storing answers anywhere
        self.controller.show("SubmitPage")


# ---------------- SUBMIT PAGE ----------------
class SubmitPage(ttk.Frame):
    def __init__(self, parent, controller):
        super().__init__(parent, padding=20)
        self.controller = controller

        ttk.Label(self, text="Submit & Start Tracking", font=("Arial", 16, "bold")).pack(pady=10)

        ttk.Button(self, text="Open Website & Start Tracking", command=self.open_and_track).pack(pady=15)
        ttk.Button(self, text="Save Activity Log", command=self.save).pack(pady=10)
        ttk.Button(self, text="Exit", command=self.quit).pack(pady=10)

    def open_and_track(self):
        # open browser
        webbrowser.open(TRACKING_URL)

        # start input tracking
        start_tracking_background()

        messagebox.showinfo("Tracking Started", "Now tracking mouse, keyboard, and idle events in background.")

    def save(self):
        save_events_to_excel()
        messagebox.showinfo("Saved", f"Events saved to {OUTPUT_FILE}")


# ---------------- RUN APP ----------------
if __name__ == "__main__":
    App().mainloop()


C:\Users\gorip\AppData\Local\Temp\ipykernel_17660\2228855071.py:47: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  ts = datetime.utcnow()
Exception in Tkinter callback
Traceback (most recent call last):
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.13_3.13.2544.0_x64__qbz5n2kfra8p0\Lib\tkinter\__init__.py", line 2074, in __call__
    return self.func(*args)
           ~~~~~~~~~^^^^^^^
  File "C:\Users\gorip\AppData\Local\Temp\ipykernel_17660\2228855071.py", line 216, in save
    save_events_to_excel()
    ~~~~~~~~~~~~~~~~~~~~^^
  File "C:\Users\gorip\AppData\Local\Temp\ipykernel_17660\2228855071.py", line 107, in save_events_to_excel
    df.to_excel(OUTPUT_FILE, index=False)
    ~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\gorip\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kf

: 

In [2]:
# app.py
"""
Local Webpage + Tkinter UI + Background Input Tracking

Flow:
 - Login page (enter participant id)         [participant id NOT saved]
 - Questions page (6 questions)              [answers NOT saved]
 - Submit page:
     - Creates and serves a small local webpage (avoids external permission issues)
     - Opens that webpage in browser
     - Starts background tracking of mouse clicks, key presses, and idle periods
 - Save only the activity events to activity_log.xlsx
   Columns: timestamp, event_type, detail, x, y

Dependencies:
    pip install pynput pandas openpyxl

Run:
    python app.py
"""

import os
import threading
import time
import socket
import tempfile
import webbrowser
from datetime import datetime
from http.server import HTTPServer, SimpleHTTPRequestHandler
from pathlib import Path
import tkinter as tk
from tkinter import ttk, messagebox
import pandas as pd

# pynput imports (required)
from pynput import mouse, keyboard

# ---------- CONFIG ----------
OUTPUT_FILE = "activity_log.xlsx"
IDLE_THRESHOLD = 5.0  # seconds
LOCALPAGE_FILENAME = "local_portal.html"
# ----------------------------

# Global state for events
event_log = []
_last_event_time = None
_tracking_started = False
_server_thread = None
_httpd = None
_server_dir = None


# ---------- Input event recording ----------
def _now_iso():
    return datetime.utcnow().isoformat()


def record_idle_if_needed(now_ts):
    global _last_event_time
    if _last_event_time is None:
        _last_event_time = now_ts
        return
    gap = (now_ts - _last_event_time).total_seconds()
    if gap > IDLE_THRESHOLD:
        event_log.append({
            "timestamp": now_ts.isoformat(),
            "event_type": "idle",
            "detail": f"idle_for_{gap:.2f}_seconds",
            "x": None,
            "y": None
        })
    _last_event_time = now_ts


def _record_event(event_type, detail=None, x=None, y=None):
    now_ts = datetime.utcnow()
    # first detect and record idle if big gap since last event
    record_idle_if_needed(now_ts)
    # then record actual event
    event_log.append({
        "timestamp": now_ts.isoformat(),
        "event_type": event_type,
        "detail": detail,
        "x": x,
        "y": y
    })


def on_click(x, y, button, pressed):
    # only record clicks (pressed True)
    if pressed:
        _record_event("mouse_click", str(button), x, y)


def on_scroll(x, y, dx, dy):
    # optionally record scrolls as well
    _record_event("mouse_scroll", f"dx={dx},dy={dy}", x, y)


def on_press(key):
    try:
        k = key.char
    except AttributeError:
        k = str(key)
    _record_event("key_press", k, None, None)


# ---------- Background tracking ----------
def start_input_listeners():
    """Start pynput listeners in daemon threads."""
    # If already started, don't start again
    global _tracking_started
    if _tracking_started:
        return
    _tracking_started = True

    # Start mouse listener
    mouse_listener = mouse.Listener(on_click=on_click, on_scroll=on_scroll)
    mouse_listener.daemon = True
    mouse_listener.start()

    # Start keyboard listener
    kb_listener = keyboard.Listener(on_press=on_press)
    kb_listener.daemon = True
    kb_listener.start()


# ---------- Local HTTP server (serves one page) ----------
def find_free_port():
    """Return a free port by binding to port 0 temporarily."""
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.bind(("", 0))
        return s.getsockname()[1]


class SilentHandler(SimpleHTTPRequestHandler):
    """Simple handler that suppresses logging to console."""
    def log_message(self, format, *args):
        pass


def serve_local_page(directory, port):
    """Start a simple HTTP server serving 'directory' at given port."""
    global _httpd
    os.chdir(directory)  # serve from this directory
    handler = SilentHandler
    _httpd = HTTPServer(("localhost", port), handler)
    try:
        _httpd.serve_forever()
    except Exception:
        pass


def start_local_server_with_page(html_content):
    """
    Create a temporary directory, write html_content to LOCALPAGE_FILENAME,
    start a HTTP server in background on a free port, and return the URL.
    """
    global _server_thread, _server_dir
    port = find_free_port()
    tmpdir = tempfile.TemporaryDirectory(prefix="local_portal_")
    _server_dir = tmpdir  # keep reference so it's not deleted
    dpath = Path(tmpdir.name)
    page_path = dpath / LOCALPAGE_FILENAME
    page_path.write_text(html_content, encoding="utf-8")

    # start server thread
    _server_thread = threading.Thread(target=serve_local_page, args=(str(dpath), port), daemon=True)
    _server_thread.start()

    # Wait briefly to ensure server starts
    time.sleep(0.5)
    url = f"http://localhost:{port}/{LOCALPAGE_FILENAME}"
    return url


# ---------- Save events ----------
def save_events_to_excel():
    if not event_log:
        # nothing to save, create an empty file or warn
        df = pd.DataFrame(columns=["timestamp", "event_type", "detail", "x", "y"])
    else:
        df = pd.DataFrame(event_log)

    # If file exists, append
    if os.path.exists(OUTPUT_FILE):
        try:
            existing = pd.read_excel(OUTPUT_FILE)
            combined = pd.concat([existing, df], ignore_index=True, sort=False)
            combined.to_excel(OUTPUT_FILE, index=False)
        except Exception as e:
            # fallback: write a new timestamped file
            alt = f"activity_log_backup_{int(time.time())}.xlsx"
            df.to_excel(alt, index=False)
            raise RuntimeError(f"Could not append to {OUTPUT_FILE}: {e}. Saved to {alt} instead.")
    else:
        df.to_excel(OUTPUT_FILE, index=False)


# ---------- Sample local HTML page ----------
LOCAL_HTML = """
<!doctype html>
<html>
<head>
  <meta charset="utf-8"/>
  <title>Local Student Portal (Demo)</title>
  <style>
    body { font-family: Arial, sans-serif; margin: 30px; }
    header { margin-bottom: 20px; }
    .box { border: 1px solid #ddd; padding: 12px; border-radius: 6px; max-width:700px; }
    input, textarea { width: 100%; padding: 8px; margin:6px 0; box-sizing: border-box; }
    button { padding: 10px 14px; }
  </style>
</head>
<body>
  <header>
    <h1>Student Portal (Local Demo)</h1>
    <p>This is a local demo page served from your machine. Use it to simulate user interaction.</p>
  </header>

  <section class="box">
    <h3>Login</h3>
    <label>Username</label>
    <input id="user" placeholder="username"/>
    <label>Password</label>
    <input id="pwd" type="password" placeholder="password"/>
    <button onclick="alert('Logged in (demo)')">Login</button>
  </section>

  <section style="height:20px"></section>

  <section class="box">
    <h3>Form</h3>
    <label>Feedback</label>
    <textarea id="feedback" rows="4"></textarea>
    <button onclick="alert('Feedback submitted (demo)')">Submit Feedback</button>
  </section>

  <section style="height:20px"></section>

  <section class="box">
    <h3>Sample Links</h3>
    <a href="#" onclick="alert('Link clicked')">Profile</a> |
    <a href="#" onclick="alert('Link clicked')">Grades</a>
  </section>

  <footer style="margin-top:30px; color:#666">
    <small>Local demo page. Close the browser tab when finished and click 'Save Activity Log' in the app.</small>
  </footer>
</body>
</html>
"""


# ---------- Tkinter UI ----------
class App(tk.Tk):
    def __init__(self):
        super().__init__()
        self.title("Activity Capture App (Local Page)")
        self.geometry("660x520")
        self.protocol("WM_DELETE_WINDOW", self.on_close)

        container = ttk.Frame(self)
        container.pack(fill="both", expand=True)
        self.frames = {}

        for F in (LoginPage, QuestionsPage, SubmitPage):
            page = F(container, self)
            self.frames[F.__name__] = page
            page.grid(row=0, column=0, sticky="nsew")

        self.show_frame("LoginPage")

    def show_frame(self, name):
        frame = self.frames[name]
        frame.tkraise()

    def on_close(self):
        if messagebox.askokcancel("Quit", "Quit application?"):
            # stop server if running
            try:
                if _httpd:
                    _httpd.shutdown()
            except Exception:
                pass
            # Save events automatically (best-effort)
            try:
                save_events_to_excel()
            except Exception:
                pass
            self.destroy()


class LoginPage(ttk.Frame):
    def __init__(self, parent, controller):
        super().__init__(parent, padding=18)
        self.controller = controller
        ttk.Label(self, text="Login", font=("Helvetica", 18, "bold")).pack(pady=8)
        ttk.Label(self, text="Participant ID (not saved):").pack(anchor="w")
        self.pid_var = tk.StringVar()
        ttk.Entry(self, textvariable=self.pid_var, width=40).pack(pady=6)
        ttk.Button(self, text="Next", command=self.next).pack(pady=12)

    def next(self):
        if not self.pid_var.get().strip():
            messagebox.showerror("Missing", "Please enter a Participant ID (this will not be saved).")
            return
        self.controller.show_frame("QuestionsPage")


QUESTIONS = [
    "1. What is your age group?",
    "2. Have you used similar apps before? (Yes/No)",
    "3. Rate your familiarity with computers (1-5).",
    "4. How comfortable are you with typing? (1-5).",
    "5. What is your primary purpose for participating?",
    "6. Any additional comments or feedback?"
]


class QuestionsPage(ttk.Frame):
    def __init__(self, parent, controller):
        super().__init__(parent, padding=12)
        self.controller = controller
        ttk.Label(self, text="Questions (Not saved)", font=("Helvetica", 16, "bold")).pack(pady=6)

        self.entries = []
        for q in QUESTIONS:
            frame = ttk.Frame(self)
            frame.pack(fill="x", pady=4)
            ttk.Label(frame, text=q).pack(anchor="w")
            ent = ttk.Entry(frame)
            ent.pack(fill="x")
            self.entries.append(ent)

        ttk.Button(self, text="Submit and Open Local Page", command=self.go_submit).pack(pady=12)

    def go_submit(self):
        # We intentionally do NOT store answers anywhere
        self.controller.show_frame("SubmitPage")


class SubmitPage(ttk.Frame):
    def __init__(self, parent, controller):
        super().__init__(parent, padding=14)
        self.controller = controller
        ttk.Label(self, text="Submit & Start Tracking", font=("Helvetica", 16, "bold")).pack(pady=6)

        ttk.Label(self, text="After clicking 'Open Local Page', a browser tab will open showing a local demo portal.\nThe app will begin tracking mouse clicks, key presses and idle periods (background).").pack(pady=8)

        btn_frame = ttk.Frame(self)
        btn_frame.pack(pady=12)

        ttk.Button(btn_frame, text="Open Local Page & Start Tracking", command=self.open_local_and_start).pack(side="left", padx=6)
        ttk.Button(btn_frame, text="Save Activity Log", command=self.save).pack(side="left", padx=6)
        ttk.Button(btn_frame, text="Quit (Saves)", command=self.quit_app).pack(side="left", padx=6)

        # summary box
        self.summary = tk.Text(self, height=12)
        self.summary.pack(fill="both", expand=True, pady=10)
        self.refresh_summary()

        # refresh every 2 seconds for event count
        self.after(2000, self.periodic_refresh)

    def open_local_and_start(self):
        try:
            url = start_local_server_with_page(LOCAL_HTML)
        except Exception as e:
            messagebox.showerror("Server error", f"Could not start local server: {e}")
            return

        # open default browser to local page
        webbrowser.open(url)

        # start background tracking listeners
        start_input_listeners()

        messagebox.showinfo("Tracking started", f"Local page opened at:\n{url}\nTracking mouse, keyboard, idle events now.")

    def save(self):
        try:
            save_events_to_excel()
            messagebox.showinfo("Saved", f"Activity log saved to {OUTPUT_FILE}")
            self.refresh_summary()
        except Exception as e:
            messagebox.showerror("Save error", str(e))

    def quit_app(self):
        # Save and quit
        try:
            save_events_to_excel()
        except Exception:
            pass
        self.controller.on_close()

    def refresh_summary(self):
        self.summary.delete("1.0", "end")
        self.summary.insert("1.0", f"Captured events: {len(event_log)}\n")
        if event_log:
            last = event_log[-1]
            self.summary.insert("end", f"Last event: {last['timestamp']} | {last['event_type']} | {last.get('detail')}\n")
        self.summary.insert("end", "\nNote: Questions / Answers are NOT recorded in the Excel file.\n")

    def periodic_refresh(self):
        self.refresh_summary()
        self.after(2000, self.periodic_refresh)


# ---------- main ----------
def main():
    app = App()
    app.mainloop()


if __name__ == "__main__":
    main()


Exception ignored in: <finalize object at 0x21d409e6fe0; dead>
Traceback (most recent call last):
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.13_3.13.2544.0_x64__qbz5n2kfra8p0\Lib\weakref.py", line 590, in __call__
    return info.func(*info.args, **(info.kwargs or {}))
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.13_3.13.2544.0_x64__qbz5n2kfra8p0\Lib\tempfile.py", line 940, in _cleanup
    cls._rmtree(name, ignore_errors=ignore_errors)
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.13_3.13.2544.0_x64__qbz5n2kfra8p0\Lib\tempfile.py", line 935, in _rmtree
    _shutil.rmtree(name, onexc=onexc)
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.13_3.13.2544.0_x64__qbz5n2kfra8p0\Lib\shutil.py", line 790, in rmtree
    return _rmtree_unsafe(path, onexc)
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.13_3.13.2544.0_x64__qbz5n2kfra8p0\Lib\shutil.py", line 635, in _rmtree_un

: 

In [ ]:
import tkinter as tk
from tkinter import ttk, messagebox
from datetime import datetime
import threading
import time
import webbrowser
from http.server import SimpleHTTPRequestHandler, HTTPServer
import pandas as pd
import os

from pynput import mouse, keyboard

# ===================== CONFIG =====================
OUTPUT_FILE = "activity_logger.xlsx"
IDLE_THRESHOLD = 5.0
PORTAL_URL = "http://localhost:8000"
TRACKING_STARTED = False
event_log = []
_last_event_time = None
# ==================================================


# ---------------- IDLE + EVENT LOGGING ----------------
def record_event(event_type, detail=None, x=None, y=None):
    global _last_event_time
    
    ts = datetime.utcnow()

    if _last_event_time:
        gap = (ts - _last_event_time).total_seconds()
        if gap > IDLE_THRESHOLD:
            event_log.append({
                "timestamp": ts.isoformat(),
                "event_type": "idle",
                "detail": f"idle_for_{gap:.2f}_seconds",
                "x": None,
                "y": None
            })

    _last_event_time = ts

    event_log.append({
        "timestamp": ts.isoformat(),
        "event_type": event_type,
        "detail": detail,
        "x": x,
        "y": y
    })


def on_click(x, y, button, pressed):
    if pressed:
        record_event("mouse_click", str(button), x, y)


def on_press(key):
    try:
        detail = key.char
    except:
        detail = str(key)
    record_event("key_press", detail)


def start_tracking():
    global TRACKING_STARTED
    if TRACKING_STARTED:
        return
    
    TRACKING_STARTED = True

    threading.Thread(target=lambda: mouse.Listener(on_click=on_click).run(), daemon=True).start()
    threading.Thread(target=lambda: keyboard.Listener(on_press=on_press).run(), daemon=True).start()


# ---------------- SAVE TO EXCEL ----------------
def save_to_excel():
    df = pd.DataFrame(event_log)

    if os.path.exists(OUTPUT_FILE):
        old = pd.read_excel(OUTPUT_FILE)
        df = pd.concat([old, df], ignore_index=True)

    df.to_excel(OUTPUT_FILE, index=False)


# ---------------- LOCAL HOST WEBPAGE ----------------
def start_local_server():
    handler = SimpleHTTPRequestHandler
    server = HTTPServer(("localhost", 8000), handler)

    threading.Thread(target=server.serve_forever, daemon=True).start()


# =====================================================
#                   TKINTER UI
# =====================================================

class App(tk.Tk):
    def __init__(self):
        super().__init__()
        self.title("Tracking App")
        self.geometry("600x420")

        self.container = ttk.Frame(self)
        self.container.pack(fill="both", expand=True)

        self.frames = {}

        for F in (LoginPage, QuestionsPage, SubmitPage):
            frame = F(parent=self.container, controller=self)
            self.frames[F.__name__] = frame
            frame.grid(row=0, column=0, sticky="nsew")

        self.show("LoginPage")

    def show(self, name):
        frame = self.frames[name]
        frame.tkraise()


# ---------------- LOGIN PAGE ----------------
class LoginPage(ttk.Frame):
    def __init__(self, parent, controller):
        super().__init__(parent, padding=20)
        self.controller = controller

        ttk.Label(self, text="Login", font=("Arial", 18)).pack(pady=10)

        ttk.Label(self, text="Participant ID:").pack()
        self.pid = tk.StringVar()
        ttk.Entry(self, textvariable=self.pid).pack(pady=5)

        ttk.Label(self, text="Password:").pack()
        self.pwd = tk.StringVar()
        ttk.Entry(self, textvariable=self.pwd, show="*").pack(pady=5)

        ttk.Button(self, text="Login", command=self.login).pack(pady=20)

    def login(self):
        if not self.pid.get().strip() or not self.pwd.get().strip():
            messagebox.showerror("Error", "Enter ID and Password")
            return

        # Launch local website
        webbrowser.open(PORTAL_URL)

        self.controller.show("QuestionsPage")


# ---------------- QUESTIONS PAGE ----------------
class QuestionsPage(ttk.Frame):
    QUESTIONS = [
        "1. What is your age group?",
        "2. Have you used similar apps before?",
        "3. Rate your computer knowledge (1-5).",
        "4. How fast do you type? (1-5)",
        "5. Reason for participation?",
        "6. Additional feedback?"
    ]

    def __init__(self, parent, controller):
        super().__init__(parent, padding=15)
        self.controller = controller

        ttk.Label(self, text="Questions", font=("Arial", 16)).pack()

        self.fields = []
        for q in self.QUESTIONS:
            ttk.Label(self, text=q).pack(anchor="w")
            entry = ttk.Entry(self)
            entry.pack(fill="x", pady=5)
            self.fields.append(entry)

        ttk.Button(self, text="Next", command=self.goto_submit).pack(pady=20)

    def goto_submit(self):
        # Do NOT save answers
        self.controller.show("SubmitPage")


# ---------------- SUBMIT PAGE ----------------
class SubmitPage(ttk.Frame):
    def __init__(self, parent, controller):
        super().__init__(parent, padding=20)
        self.controller = controller

        ttk.Label(self, text="Submit", font=("Arial", 18)).pack(pady=10)

        ttk.Button(self, text="Start Tracking Activities", command=self.start_tracking_activity).pack(pady=15)
        ttk.Button(self, text="Save Log to Excel", command=self.save_log).pack(pady=10)
        ttk.Button(self, text="Exit", command=self.quit).pack(pady=10)

    def start_tracking_activity(self):
        start_tracking()
        messagebox.showinfo("Tracking", "Now tracking mouse, keyboard, and idle time.")

    def save_log(self):
        save_to_excel()
        messagebox.showinfo("Saved", "Activity Log saved successfully!")


# ========================== RUN ==============================
if __name__ == "__main__":
    start_local_server()  # start local website
    App().mainloop()


127.0.0.1 - - [23/Nov/2025 17:28:27] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [23/Nov/2025 17:28:28] code 404, message File not found
127.0.0.1 - - [23/Nov/2025 17:28:28] "GET /favicon.ico HTTP/1.1" 404 -
127.0.0.1 - - [23/Nov/2025 17:28:35] "GET /DMart_Day1_Python_Basics.ipynb HTTP/1.1" 200 -
C:\Users\gorip\AppData\Local\Temp\ipykernel_38184\540039679.py:27: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  ts = datetime.utcnow()
